In [0]:
data = [
    (1, "Alice", "NY", 5000),
    (2, "Bob", "CA", 4000),
    (3, "Alice", "NY", 5500), 
    (4, "Charlie", "TX", 3000),
    (5, "Bob", "CA", 4200),  
    (6, "Alice", "CA", 6000), 
    (7, "Alice", "NY", 5200),
    (8, "Bob", "CA", 4500)
]

columns = ["id", "name", "state", "salary"]

# Create input DataFrame
df = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql.functions import *
data = [
    (1, "Alice", "NY", 5000, "2024-01-10"),
    (2, "Alice", "NY", 5500, "2024-01-15"),  
    (3, "Bob", "CA", 4000, "2024-02-01"),
    (4, "Bob", "CA", 4200, "2024-02-20"),  
    (5, "Bob", "CA", 3900, "2024-01-05"),
    (6, "Charlie", "TX", 3000, "2024-01-12")
]

columns = ["id", "name", "state", "salary", "update_date"]

# 1. Create source DataFrame
df = spark.createDataFrame(data, columns).withColumn(
    "update_date", col("update_date").cast("date")
)

##### 1. Remove Simple Duplicate Rows

In [0]:
# Remove duplicate rows based on key columns
df_no_duplicates = df.dropDuplicates(["name", "state"])
df_no_duplicates.show()

##### 2. Deduplicate and Keep Latest Record
- When duplicate records exist, it is often necessary to retain only the most recent
version based on timestamps. 
- Window functions allow ranking rows within
partitions using orderBy(). 
- The row_number() function assigns rank and enables
filtering of latest records. This approach ensures accurate historical tracking and
correct version control in ETL pipelines.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

windowSpec = Window.partitionBy("name","state").orderBy(col("update_date").desc())

df_no_duplicates = df.withColumn("row_number", row_number().over(windowSpec)).filter("row_number ==1").drop("row_number")

display(df_no_duplicates)

##### 3. Replace Nulls using Case Logic

In [0]:
data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "Engineering", None),      
    (3, "Charlie", "Marketing", 4200),
    (4, "Diana", "Finance", None),     
    (5, "Evan", "Engineering", 6000)
]

columns = ["emp_id", "emp_name", "department", "salary"]

# Create input DataFrame
df = spark.createDataFrame(data, columns)

In [0]:
display(df)

In [0]:
df_replaced = df.withColumn("salary", when(col("salary").isNull(),0).otherwise(col("salary")))

In [0]:
display(df_replaced)

##### 4. Type-Based Bulk NULL Replacement
- Type-based replacement automatically handles null values based on column data
types. 
- This avoids hardcoding column names and improves scalability. 
- Numeric columns can be filled with 0, while string columns can be filled with default text. This
approach is highly efficient for large enterprise datasets.

In [0]:
%skip
from pyspark.sql.types import StringType, NumericType
string_cols = [
f.name for f in df.schema.fields
if isinstance(f.dataType, StringType)
]
numeric_cols = [
f.name for f in df.schema.fields
if isinstance(f.dataType, NumericType)
]
df_filled = df.fillna(
0,
subset=numeric_cols
).fillna(
"Unknown",
subset=string_cols
)
df_filled.show()